# OpenPlaque — Multivessel Research Summary v1

Consolidates the locked/feasible coronary research measurements without changing anatomy or retuning science. **Runtime → Run all**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'OpenPlaque_Multivessel_Research_Summary_v1'
REUSE_VALID_CACHES = True
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE and OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started','time':time.time()}, indent=2))
print('Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)


In [ ]:
import os, shutil, sys
os.chdir('/content')
REPO = Path('/content/OpenPlaque_multivessel_summary')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH = 'openplaque-multivessel-research-summary-from-main'
PINNED_SCIENCE_COMMIT = '28abb2ce32fe65373ef0a8be10634c4dfcf37e5e'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q /content/OpenPlaque_multivessel_summary
for k in list(sys.modules):
    if k == 'openplaque' or k.startswith('openplaque.'):
        del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.multivessel_research_summary_v1 import synthetic_self_test
print('Synthetic self-test:', synthetic_self_test())
!pytest -q /content/OpenPlaque_multivessel_summary/tests/test_multivessel_research_summary_v1.py


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'RCA_Plaque_PCAT_Research_Lock_v1/summary.json',
    DRIVE_ROOT/'LAD_Source_Space_PCAT_Feasibility_v1/summary.json',
    DRIVE_ROOT/'LAD_Distal_Reference_Plaque_Self_Calibration_v1/summary.json',
    DRIVE_ROOT/'LCX_Structural_Source_QC_Freeze_v1/summary.json',
    DRIVE_ROOT/'LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1/summary.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n' + '\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'status':'complete','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE,'required_count':len(required)}, indent=2))
print('Preflight complete:', len(required), 'required artifacts found')


In [ ]:
from openplaque.multivessel_research_summary_v1 import run
try:
    result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
    print(json.dumps(result['summary']['current_quantitative_state'], indent=2, default=str))
    print('Report:', result['report'])
    print('ZIP:', result['zip'])
except Exception as e:
    (OUTPUT/'notebook_failure.json').write_text(json.dumps({'status':'FAILED','type':type(e).__name__,'message':str(e)}, indent=2))
    raise
